In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import (
    ArrayType, DoubleType, LongType, MapType, StringType, StructField, StructType,
)

dbutils.widgets.text("ingest_date", "2026-11-06")
ingest_date = dbutils.widgets.get("ingest_date")

source_path = f"/Volumes/workspace/bank_lakehouse/landing/osm_banks/ingest_date={ingest_date}/banks.json"
table_name = "workspace.bank_lakehouse.bronze_osm_banks"

element_schema = StructType([
    StructField("type", StringType()),
    StructField("id", LongType()),
    StructField("lat", DoubleType()),
    StructField("lon", DoubleType()),
    StructField("center", StructType([
        StructField("lat", DoubleType()),
        StructField("lon", DoubleType()),
    ])),
    StructField("tags", MapType(StringType(), StringType())),
])

file_schema = StructType([
    StructField("osm3s", StructType([StructField("timestamp_osm_base", StringType())])),
    StructField("elements", ArrayType(element_schema)),
])

raw = (
    spark.read
    .schema(file_schema)
    .option("multiLine", True)
    .json(source_path)
)

bronze = (
    raw.select(
        F.explode("elements").alias("e"),
        F.col("osm3s.timestamp_osm_base").alias("_osm_snapshot_at"),
        F.col("_metadata.file_path").alias("_source_file"),
    )
    .select(
        F.col("e.type").alias("osm_type"),
        F.col("e.id").alias("osm_id"),
        F.col("e.lat").alias("lat"),
        F.col("e.lon").alias("lon"),
        F.col("e.center.lat").alias("center_lat"),
        F.col("e.center.lon").alias("center_lon"),
        F.col("e.tags").alias("tags"),
        "_osm_snapshot_at",
        "_source_file",
        F.current_timestamp().alias("_ingested_at"),
        F.lit(ingest_date).cast("date").alias("_ingest_date"),
    )
)

writer = bronze.write.format("delta").partitionBy("_ingest_date")

if spark.catalog.tableExists(table_name):
    (
        writer.mode("overwrite")
        .option("replaceWhere", f"_ingest_date = '{ingest_date}'")
        .saveAsTable(table_name)
    )
else:
    writer.mode("overwrite").saveAsTable(table_name)

print("Table écrite :", table_name)

In [0]:
df = spark.table(table_name).filter(F.col("_ingest_date") == ingest_date)

df.agg(
    F.count("*").alias("lignes"),
    F.countDistinct("osm_type", "osm_id").alias("agences_distinctes"),
    F.sum(F.col("tags").isNull().cast("int")).alias("sans_tags"),
).show()

df.groupBy("osm_type").count().show()

In [0]:
spark.sql(f"DESCRIBE HISTORY {table_name}") \
    .select("version", "timestamp", "operation") \
    .orderBy("version") \
    .show(truncate=False)